In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [2]:
# Plain LLM

def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [3]:
# testing the black box, where text comes in and text comes out

llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What’s going on?'

In [3]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Yes, probably — but it depends on the course’s enrollment rules and whether it’s still open.

If you want, I can help you figure out the best next step:
- check whether late enrollment is allowed,
- draft a message to the instructor/admin,
- or help you ask in a polite way.

A simple message could be:

> Hi, I just discovered this course and I’m very interested in joining. Is it still possible to enroll at this point? Thank you.

If you tell me the course name or platform, I can help you tailor the message.


In [ ]:
'''
The LLM gives a generic answer. It might say "you can usually join" or "check the course website." 
It doesn't know about our specific Zoomcamp courses, their enrollment policies, or their schedules. 
It tries to be helpful, but has no idea about actual enrollment status or policies.

This is different from a question like "how do I cook salmon?" - the LLM knows the answer because cooking salmon is common knowledge. 
But our courses are not in the training data.
'''

In [4]:
# Adding some context manually:

context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [5]:
# now buiding a prompt that takes into account the context

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [ ]:
"""
RAG stands for Retrieval-Augmented Generation. Generation is the LLM producing text, and retrieval is search. 
We retrieve relevant documents from our knowledge base and use them to augment what the LLM generates. 
That search step is what gives the LLM the context it needs to answer correctly.

What we just did was naive. I knew in advance which FAQ entry held the answer and pasted it in by hand. 
What we want instead is to perform search automatically. We take the student's question, find the most relevant documents, and send those to the LLM.
"""

In [6]:
# putting this idea in code, it will look something like:

def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
"""
The LLM only sees the documents we hand it, so its answers are grounded in our data. If the right document is retrieved, the answer is good. 
If it's not, the LLM gets the wrong context and the answer is wrong. Your model is only as good as your retrieval, 
so search quality matters a lot for RAG.
"""

In [7]:
# The dataset for this exercise

import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

#This returns a list of courses. Each course has a path field that points to its FAQ data.

In [8]:
# fecthing all FAQ documents from all courses:

documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [12]:
# looking at one entry:

documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [ ]:
# searching engine:

score = sim(query, document)

In [ ]:
"""
For each document in the database, you compute this score. Then you rank all documents by score and return the top N. 
What makes a search engine different from another search engine is what sim actually computes.

* text/lexical search (covered in this section): sim counts how many words the query and the document share. 
It looks at the surface form, the actual words, and matches them exactly.

* vector/semantic search (covered in module 2): sim compares the meaning of the query and the document. Same function, different similarity measure.
"""

In [ ]:
"""
Searching matters because we have around 1100 documents. Sending all of them to the LLM would be expensive and slow. 
The model would get confused with too much data. Search finds the most relevant documents to send instead.
"""

In [9]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [10]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

"""
We used boost_dict to give the question field more weight (2.0 instead of the default 1.0) and section less weight (0.5). 
Query words appearing in the question field is a stronger signal than them appearing in the section name.
We used filter_dict to only return results from the LLM Zoomcamp course. Without this filter, we'd get results from all four courses.
"""

"\nWe used boost_dict to give the question field more weight (2.0 instead of the default 1.0) and section less weight (0.5). \nQuery words appearing in the question field is a stronger signal than them appearing in the section name.\nWe used filter_dict to only return results from the LLM Zoomcamp course. Without this filter, we'd get results from all four courses.\n"

In [8]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [11]:
# minsearch supports field boosting to reflect this fact that the question fields weight more than section fields:

results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

# This is the same boosting mechanism used by Elasticsearch and Lucene.

In [12]:
# Sometimes you want to restrict the search to a specific course. minsearch supports keyword filtering:

results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [13]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [14]:
# Wrapping it in a function:

def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
search_results = search(question)

TypeError: 'list' object is not callable

In [15]:
# Instructions (also called the system prompt): this tells the LLM how to behave. It never changes, so it's the same for every request.
# The instructions tell the LLM its role and how to answer:

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [16]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [17]:
# Building the context:

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [18]:
# Building the prompt:

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [19]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [20]:
# LLM

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [21]:
# the response:

response.output[0]

ResponseOutputMessage(id='msg_06714ed9a1374605006a399033cd2481a291d08f4e272c8fe6', content=[ResponseOutputText(annotations=[], text='Yes — you can join now.\n\nYou can start learning and submitting homework as long as the submission form is still open. If you want to receive a certificate, make sure you submit your project while the course is still accepting submissions.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [22]:
response.output[0].content[0].text

'Yes — you can join now.\n\nYou can start learning and submitting homework as long as the submission form is still open. If you want to receive a certificate, make sure you submit your project while the course is still accepting submissions.'

In [23]:
response.output_text

'Yes — you can join now.\n\nYou can start learning and submitting homework as long as the submission form is still open. If you want to receive a certificate, make sure you submit your project while the course is still accepting submissions.'

In [24]:
response.usage

ResponseUsage(input_tokens=480, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=50, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=530)

In [25]:
# Calculating the price:

"""
For model gpt-5.4-mini:
Input: $0.75 per million tokens
Output: $4.50 per million tokens
"""

input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000585

In [26]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

"""
OpenAI accepts both developer and system for the instruction role. 
There may be some difference between them, but in practice I don't see it change the result either way. We use developer in this course.
"""

In [27]:
# Updated LLM function:

def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [28]:
# Full RAG:

def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [29]:
# testing:

answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, but if you want a certificate, you need to submit your project while submissions are still open.


In [30]:
rag("How do I get a certificate?")

'To get a certificate, you need to finish the course with a **live cohort** and **pass the Capstone project**.\n\nA few important notes:\n- **Self-paced mode does not include a certificate.**\n- **Homework is not mandatory** for the certificate.\n- You must also **peer-review 3 capstones** during the live course period.\n\nIf you want, I can also tell you how to make sure your name appears correctly on the certificate.'